# Experimental Data Preprocessing

This notebook preprocesses Tongji degradation trajectories from raw pickle files and saves a public-code-friendly long-format capacity table.

Expected input:
- `data/tongji/raw/*.pkl`

Generated output:
- `data/tongji/tongji_capacity_trajectories.csv`


## Module 1. Configuration


In [ ]:
from __future__ import annotations

import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.signal import savgol_filter

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TONGJI_DATA_ROOT = PROJECT_ROOT / "data" / "tongji"
RAWDATA_DIR = TONGJI_DATA_ROOT / "raw"
OUTPUT_CSV = TONGJI_DATA_ROOT / "tongji_capacity_trajectories.csv"
CURRENT_THRESHOLD_A = -0.05
EXCLUDE_GROUPS = set()

TONGJI_DATA_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Raw Tongji folder: {RAWDATA_DIR}")
print(f"Output CSV: {OUTPUT_CSV}")


## Module 2. Helper functions

The extraction here follows the same capacity logic as the Tongji `check_QV` workflow for the degradation curve part only:
- for each cycle, take the maximum discharge capacity
- prefer samples with negative current when current is available
- otherwise fall back to the finite maximum capacity in that cycle
- apply the same `advanced_smooth_capacity_curve(...)` smoothing used in the reference notebook before computing SOH trajectories


In [ ]:
def advanced_smooth_capacity_curve(
    capacity_values,
    smooth_method="none",
    smooth_window=11,
    smooth_polyorder=3,
    smooth_window_ratio=20,
    spike_abs=0.20,
    spike_k=6.0,
):
    values = np.asarray(capacity_values, dtype=float)
    if values.size == 0 or smooth_method != "savgol":
        return values

    finite_mask = np.isfinite(values)
    if not np.any(finite_mask):
        return values

    filled = values.copy()
    x = np.arange(filled.size)
    if not np.all(finite_mask):
        filled[~finite_mask] = np.interp(x[~finite_mask], x[finite_mask], filled[finite_mask])

    fixed = filled.copy()
    for index in range(1, fixed.size - 1):
        local_values = filled[index - 1:index + 2]
        local_median = float(np.nanmedian(local_values))
        local_mad = float(np.nanmedian(np.abs(local_values - local_median)))
        local_std = float(np.nanstd(local_values))
        local_scale = local_mad if np.isfinite(local_mad) and local_mad > 1e-12 else local_std
        local_threshold = max(spike_abs, spike_k * local_scale) if np.isfinite(local_scale) else spike_abs
        if np.isfinite(filled[index]) and np.isfinite(local_median) and abs(filled[index] - local_median) > local_threshold:
            left_value = filled[index - 1]
            right_value = filled[index + 1]
            if np.isfinite(left_value) and np.isfinite(right_value):
                fixed[index] = 0.5 * (left_value + right_value)
            else:
                fixed[index] = local_median

    if smooth_window_ratio and smooth_window_ratio > 0:
        window = int(fixed.size / smooth_window_ratio)
    else:
        window = int(smooth_window)
    if window % 2 == 0:
        window += 1
    window = min(window, fixed.size)
    if window % 2 == 0:
        window -= 1

    if window < max(3, smooth_polyorder + 2):
        return fixed

    try:
        return savgol_filter(fixed, window, smooth_polyorder, mode="interp")
    except Exception:
        return fixed


def dataset_nominal_capacity_ah(cell_name: str) -> float:
    if cell_name.startswith(("Tongji1_", "Tongji2_")):
        return 3.5
    if cell_name.startswith("Tongji3_"):
        return 2.5
    return float("nan")


def dataset_subgroup(cell_name: str) -> str | None:
    stem = Path(cell_name).stem
    for prefix in [
        "Tongji1_CY25-1",
        "Tongji1_CY25-05",
        "Tongji1_CY25-025",
        "Tongji1_CY35-05",
        "Tongji1_CY45-05",
        "Tongji2_CY25-05",
        "Tongji2_CY35-05",
        "Tongji2_CY45-05",
        "Tongji3_CY25-05",
    ]:
        if stem.startswith(prefix):
            return prefix
    return None


def load_pickle_cycles(path: Path):
    with open(path, "rb") as f:
        obj = pickle.load(f)
    cycle_rows = obj["cycle_data"] if isinstance(obj, dict) and "cycle_data" in obj else obj
    return cycle_rows


def extract_cycle_capacity_ah(cycle_data: dict, current_threshold: float = CURRENT_THRESHOLD_A) -> float:
    discharge_capacity = np.asarray(cycle_data.get("discharge_capacity_in_Ah", []), dtype=float)
    if discharge_capacity.size == 0:
        return float("nan")

    current = np.asarray(cycle_data.get("current_in_A", np.full_like(discharge_capacity, np.nan)), dtype=float)
    valid = np.isfinite(discharge_capacity)
    if not np.any(valid):
        return float("nan")

    if np.any(np.isfinite(current)):
        discharge_mask = np.isfinite(current) & (current < current_threshold) & valid
        if np.sum(discharge_mask) >= 3:
            return float(np.nanmax(discharge_capacity[discharge_mask]))

    return float(np.nanmax(discharge_capacity[valid]))


def collect_cycle_capacities(cell_cycles):
    raw_capacities = []
    for cycle_data in cell_cycles:
        try:
            raw_capacities.append(extract_cycle_capacity_ah(cycle_data))
        except Exception:
            raw_capacities.append(float("nan"))
    return np.asarray(raw_capacities, dtype=float)


def smooth_capacity_trace(raw_capacities: np.ndarray) -> np.ndarray:
    return advanced_smooth_capacity_curve(
        raw_capacities,
        smooth_method="savgol",
        smooth_polyorder=3,
        smooth_window_ratio=20,
        spike_abs=0.20,
        spike_k=6.0,
    )


## Module 3. Discover Tongji raw files


In [ ]:
if not RAWDATA_DIR.exists():
    raise FileNotFoundError(RAWDATA_DIR)

raw_files = sorted(
    [p for p in RAWDATA_DIR.iterdir() if p.suffix.lower() in {".pkl", ".pickle"}]
)
if not raw_files:
    raise ValueError(f"No Tongji pickle files found in {RAWDATA_DIR}")

print(f"Indexed {len(raw_files)} Tongji raw files")
display(pd.DataFrame({"file": [p.name for p in raw_files]}).head())


## Module 4. Extract full capacity trajectories

This module creates the master long-format table. Every available cycle from 1 to the last cycle in the file is retained.

Two capacity columns are saved:
- `capacity_Ah_raw`: direct per-cycle extracted discharge capacity
- `capacity_Ah`: the smoothed capacity trajectory using the same smoothing rule as `check_QV_tongji`


In [ ]:
records = []
summary_rows = []

for raw_path in raw_files:
    cell_name = raw_path.stem
    subgroup = dataset_subgroup(cell_name)
    if subgroup in EXCLUDE_GROUPS:
        continue

    nominal_capacity = dataset_nominal_capacity_ah(cell_name)
    cycle_rows = load_pickle_cycles(raw_path)
    raw_capacities = collect_cycle_capacities(cycle_rows)
    smooth_capacities = smooth_capacity_trace(raw_capacities)

    valid_count = int(np.sum(np.isfinite(raw_capacities)))
    for cycle_idx, (capacity_raw_ah, capacity_smooth_ah) in enumerate(zip(raw_capacities, smooth_capacities), start=1):
        soh_pct = float("nan")
        if np.isfinite(nominal_capacity) and nominal_capacity > 0 and np.isfinite(capacity_smooth_ah):
            soh_pct = 100.0 * capacity_smooth_ah / nominal_capacity

        records.append({
            "cell": cell_name,
            "cycle": int(cycle_idx),
            "capacity_Ah_raw": float(capacity_raw_ah) if np.isfinite(capacity_raw_ah) else float("nan"),
            "capacity_Ah": float(capacity_smooth_ah) if np.isfinite(capacity_smooth_ah) else float("nan"),
            "nominal_capacity_Ah": float(nominal_capacity) if np.isfinite(nominal_capacity) else float("nan"),
            "soh_pct": float(soh_pct) if np.isfinite(soh_pct) else float("nan"),
            "subgroup": subgroup,
            "major_group": cell_name.split("_")[0] if "_" in cell_name else None,
            "source_file": raw_path.name,
        })

    summary_rows.append({
        "cell": cell_name,
        "subgroup": subgroup,
        "n_cycles_total": int(len(cycle_rows)),
        "n_cycles_with_capacity": int(valid_count),
        "nominal_capacity_Ah": float(nominal_capacity) if np.isfinite(nominal_capacity) else float("nan"),
        "first_capacity_Ah": float(smooth_capacities[0]) if len(smooth_capacities) and np.isfinite(smooth_capacities[0]) else float("nan"),
        "last_capacity_Ah": float(smooth_capacities[-1]) if len(smooth_capacities) and np.isfinite(smooth_capacities[-1]) else float("nan"),
    })

capacity_df = pd.DataFrame(records).sort_values(["cell", "cycle"]).reset_index(drop=True)
cell_summary_df = pd.DataFrame(summary_rows).sort_values(["subgroup", "cell"]).reset_index(drop=True)

if capacity_df.empty:
    raise ValueError("No Tongji capacity records were extracted.")

print(f"Extracted rows: {len(capacity_df)}")
print(f"Cells covered: {capacity_df['cell'].nunique()}")
print("Cell count by nominal capacity:")
print(cell_summary_df.groupby('nominal_capacity_Ah').size().to_string())
display(capacity_df.head())
display(cell_summary_df.head())


## Module 5. Quick trajectory audit


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5), constrained_layout=True)

for _, grp in capacity_df.groupby("cell"):
    axes[0].plot(grp["cycle"], grp["capacity_Ah"], alpha=0.28, linewidth=1.0, color="#2f6f9f")
axes[0].set_title("Tongji smoothed capacity trajectories")
axes[0].set_xlabel("Cycle")
axes[0].set_ylabel("Capacity (Ah)")
axes[0].grid(alpha=0.25)

for _, grp in capacity_df.groupby("cell"):
    axes[1].plot(grp["cycle"], grp["soh_pct"], alpha=0.28, linewidth=1.0, color="#b84a39")
axes[1].set_title("Tongji smoothed SOH trajectories")
axes[1].set_xlabel("Cycle")
axes[1].set_ylabel("SOH (%)")
axes[1].grid(alpha=0.25)

plt.show()

print("Cycle coverage summary:")
display(cell_summary_df.describe(include="all"))


## Module 6. Save the master CSV

The saved CSV is the recommended canonical format for downstream Tongji curve fitting and matching.


In [ ]:
capacity_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved processed Tongji trajectories to: {OUTPUT_CSV}")
print(f"CSV shape: {capacity_df.shape}")

# Optional in-notebook preview of the wide representation, without saving it as the master file.
capacity_pivot_preview = capacity_df.pivot(index="cell", columns="cycle", values="capacity_Ah")
print(f"Wide preview shape: {capacity_pivot_preview.shape}")
display(capacity_pivot_preview.iloc[:5, :8])
